# 🧠 CerebroSpheroid V5: Cloud TPU High-Throughput Engine
### 166.3M Compact Architecture (32 Concentric Shells, 96 Shortcut Tracts) • 200B Token Campaign

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_TPU_JAX.ipynb)

- **Pure JAX on Cloud TPU**: Native bfloat16 systolic MXU acceleration (~100,000–350,000 tok/s on TPU v5e / v6e).
- **Hardware Auto-Tuning**: Automatically discovers and tunes peak micro-batches and remat policy.
- **50,000-Step Cortical Maturation**: Shells 0–15 frozen for initial 50k steps, then permanently unfrozen for full end-to-end training.
- **Autonomous Streaming**: Real-time developmental streaming from FineWeb-Edu $\ge 3.0$ and foundational curricula.
- **Remote Cloud Sync**: Real-time checkpoint mirroring and loss curve tracking on `bioflo.in`.

---
### ⚡ Colab Hardware Setting:
**Runtime** -> **Change runtime type** -> **TPU v5e** (TPU v5e-1) or **TPU v6e** (Trillium)

## Step 1: Environment Setup, HF Hub Auth & Repository Sync
Installs required packages, detects Cloud TPU, logs into Hugging Face Hub (eliminating 429 rate limits), and synchronizes the latest codebase.

In [ ]:
# ==============================================================================
# 🚀 Step 1: Dependencies, Hugging Face Auth & Repository Sync
# ==============================================================================
!pip install -q transformers datasets sentencepiece optax torch pyarrow huggingface_hub

import os, sys, subprocess, jax

# 1. ⚡ Verify Cloud TPU Environment
devices = jax.devices()
dev_kind = devices[0].device_kind if devices else "CPU"
is_tpu = any("TPU" in d.device_kind.upper() for d in devices)
print(f"⚡ JAX Device: {dev_kind} ({len(devices)} device(s) active, Backend: {jax.default_backend().upper()})")
if is_tpu:
    print(f"✅ Cloud TPU Acceleration Active! ({dev_kind})")
else:
    print("⚠️ Running on CPU/GPU. For peak throughput, select Runtime -> Change runtime type -> TPU v5e.")

# 2. 🔑 Authenticate Hugging Face Hub (Eliminates 429 Too Many Requests)
try:
    from google.colab import userdata
    hf_tok = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN', '')
except Exception:
    hf_tok = os.environ.get('HF_TOKEN', '')

if hf_tok:
    os.environ['HF_TOKEN'] = hf_tok
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_tok
    try:
        from huggingface_hub import login
        login(token=hf_tok, add_to_git_credential=False)
        print("🔑 Hugging Face Hub authenticated successfully!")
    except Exception as e:
        print(f"ℹ️ Hugging Face login notice: {e}")

# 3. 🔄 Synchronize CerebroSpheroid V5 Codebase
REPO_DIR = '/content/cerebro_v5'
if not os.path.exists(REPO_DIR):
    print("📥 Cloning cerebro_spheroid_v5 repository...")
    subprocess.run(['git', 'clone', 'https://github.com/AjitKumar-git/cerebro_spheroid_v5.git', REPO_DIR], check=True)
else:
    print("🔄 Pulling latest updates from origin/main...")
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

rev = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
print(f"✅ Repository synchronized at commit [{rev}] in {REPO_DIR}!")

# 4. 📁 Ensure Checkpoint Directory Exists
CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"📁 Local Checkpoint Directory Ready: {CKPT_DIR}")


## Step 2: Configure & Launch Training
Launches pretraining for `v5_compact_32shells` (166.3M parameters, 32 concentric shells, 96 shortcut tracts).
- **Auto-Tune**: Micro-batches, remat policy, and gradient accumulation are automatically calibrated for optimal TPU MXU throughput.
- **50,000-Step Freezing**: Shells 0–15 frozen for initial 50k steps, then permanently unfrozen for full end-to-end training.
- **Resumption**: Set `FROM_SCRATCH = False` to resume automatically from existing local/cloud checkpoints.

In [ ]:
# ==============================================================================
# ⚡ Step 2: Launch Progressive Cortical Expansion Training
# ==============================================================================
from training.train_v5_tpu_jax import run_training

# ── Primary Configuration ─────────────────────────────────────────────────────
MODEL_PRESET       = 'v5_compact_32shells'   # 166.3M Parameters, 32 Concentric Shells
FROM_SCRATCH       = False                    # True: Fresh start at Step 0 | False: Resume existing checkpoint
FREEZE_CORE        = True                    # Freeze Core (Shells 0-15) for initial phase
FREEZE_UNTIL_STEP  = 50000                   # Permanently unfreezes core at Step 50,000
AUTO_TUNE          = True                    # Auto-tune decides micro-batch & remat on TPU hardware
EFFECTIVE_BATCH    = 256                     # Global effective batch size (131,072 to 196,608 tok/step)
TARGET_TOKENS      = 200_000_000_000         # 200 Billion Token Target Budget
DATASET_SOURCE     = 'autonomous'            # Online streaming: FineWeb-Edu >= 3.0 curriculum

# ── Execute High-Throughput JAX Runner ─────────────────────────────────────────
run_training(
    preset=MODEL_PRESET,
    from_scratch=FROM_SCRATCH,
    freeze_core=FREEZE_CORE,
    freeze_until_step=FREEZE_UNTIL_STEP,
    auto_tune=AUTO_TUNE,
    effective_batch_size=EFFECTIVE_BATCH,
    target_tokens=TARGET_TOKENS,
    dataset_source=DATASET_SOURCE,
    checkpoint_dir="/content/checkpoints",
    cloud_sync=True,
)


## Step 3: Monitor Checkpoints & Cloud Synchronization
Inspect local checkpoints and live sync status on the bioflo.in remote hub.

In [ ]:
# ==============================================================================
# 📊 Step 3: Checkpoint Inspector & Cloud Hub Dashboard
# ==============================================================================
import json
from src.cloud_sync import check_remote_manifest

print("📁 Local Saved Checkpoints:")
!ls -lh /content/checkpoints

print("\n🌐 Remote bioflo.in Cloud Hub Status (v5_compact):")
try:
    manifest = check_remote_manifest(model="v5_compact")
    print(json.dumps(manifest, indent=2))
except Exception as e:
    print(f"ℹ️ Cloud sync status notice: {e}")

print("\n📈 Live Web Dashboard: https://bioflo.in/cerebro")
